# E5 (2022)
---
[[paper]](https://arxiv.org/abs/2212.03533)<br>
E5 = EmbEddings from bidirectionAl Encoder representations

__E5__ — это семейство моделей для генерации текстовых эмбеддингов, которые первыми показали, что использование масштабного слабоструктурированного обучения (Weakly-supervised learning) на парах текстов в сочетании с инструкциями позволяет превзойти модели, обучавшиеся на узких размеченных датасетах.

# Задача
Решается задача Dense Retrieval. Необходимо отобразить тексты произвольной длины (запросы и документы) в векторы фиксированной размерности так, чтобы семантически близкие объекты находились рядом в векторном пространстве (обычно по Cosine Similarity).

# Мотивация
До появления E5 большинство SOTA-моделей Dense Retrieval сильно зависели от качества и объема размеченных данных (например, MS MARCO). Проблема в том, что ручная разметка пар "запрос-ответ" стоит дорого, а модели, обученные только на них, плохо обобщаются на новые домены (Zero-shot retrieval). Существовавшие методы предобучения без учителя, такие как SimCSE (2021), хорошо работали для Sentence Similarity, но проигрывали в задачах поиска, так как не учитывали асимметрию между коротким запросом и длинным документом.

# Подходы
На момент появления E5 существовали следующие решения:
- BM25: классический Sparse Retrieval — не учитывает семантику, работает только по совпадению слов.
- DPR (2020): Bi-Encoder архитектура, обученная на малом количестве золотых пар (Gold passages) с использованием Hard Negatives. Требует сложной процедуры подбора негативных примеров для сходимости.
- Contriever (2021): использует Contrastive Learning на неразмеченных данных через Inverse Cloze Task. Хорошо обобщается, но уступает в точности на специфических бенчмарках из-за отсутствия этапа Fine-tuning на качественных данных.
- SimCSE (2021): обучение на идентичных предложениях с разным Dropout. Метод оптимизирован под схожесть предложений, а не под поиск релевантной информации.

# Идея
Авторы предложили двухэтапный процесс обучения. Главная новизна заключается в создании гигантского датасета CCP (Curation of Contrastive Pairs) объемом 1.3 млрд пар, извлеченного из открытых источников (веб-графы, ссылки, посты), и введении специальных префиксов `query: ` и `passage: `. Это позволяет модели понять асимметричную природу поиска: запрос ищет ответ, а не просто похожий по структуре текст.

# Архитектура
E5 базируется на стандартной архитектуре Transformer-энкодера (использовались версии `small`, `base`, `large`, основанные на BERT (2018) или RoBERTa (2019)).
1. Input: Текст с префиксом. Для запроса — `query: {text}`, для документа — `passage: {text}`.
2. Encoder: Трансформер обрабатывает последовательность.
3. Pooling: В отличие от многих моделей, использующих `[CLS]` токен, в E5 используется Mean Pooling по всем токенам выходного слоя, что дает более стабильное представление смысла всего предложения.
4. Normalization: Выходной вектор нормализуется (L2), чтобы вычисление Cosine Similarity сводилось к простому скалярному произведению.

# Обучение
Процесс разбит на две стадии:
1. Contrastive Pre-training: Обучение на 1.3 млрд "слабых" пар из интернета. Используется InfoNCE loss. В качестве позитивного примера берется пара (заголовок, статья) или (текст, связанный текст), в качестве негативных — другие тексты в батче (In-batch negatives). Это дает модели общие знания о семантике.
2. Supervised Fine-tuning: Дообучение на смеси высококачественных наборов данных (MS MARCO, NLI, Natural Questions). Здесь модель учится различать тонкие нюансы релевантности. На этом этапе также используются Hard Negatives (документы, которые похожи на релевантные, но таковыми не являются).

# Инференс
Алгоритм применения стандартен для Bi-Encoder систем:
1. Индексация: Все документы коллекции (Passages) прогоняются через энкодер с префиксом `passage: `. Полученные векторы сохраняются в векторную БД (например, FAISS или HNSW индекс).
2. Обработка запроса: Пришедший запрос получает префикс `query: ` и переводится в вектор тем же энкодером.
3. Поиск: Выполняется поиск K ближайших соседей (K-NN) в векторном пространстве.

# Результаты
E5 показала значительный прирост на бенчмарке BEIR (Zero-shot retrieval):
- Модель E5-large достигла 51.2 балла по метрике nDCG@10, что на тот момент было на 2.4 п.п. выше, чем у предыдущего лидера Contriever, и на 7-10 п.п. выше, чем у моделей, обученных только на MS MARCO.
- Примечательно, что даже версия E5-small (всего 33M параметров) превзошла по качеству поиска модели типа DPR-base (110M параметров), что подтверждает критическую важность этапа предобучения на сверхбольших массивах данных CCP.

## 📝 Критический анализ

```markdown
# E5 (2022)
---
[[paper]](https://arxiv.org/abs/2212.03533)<br>
E5 = EmbEddings from bidirectionAl Encoder representations

__E5__ — семейство моделей для генерации текстовых эмбеддингов, которые первыми показали, что использование масштабного слабоструктурированного обучения на парах текстов с инструкциями превосходит модели, обученные на узких размеченных датасетах.

## Задача
Решается задача Dense Retrieval: отображение текстов в векторы фиксированной размерности для близости в векторном пространстве по Cosine Similarity.

## Мотивация
До E5 большинство моделей Dense Retrieval зависели от размеченных данных, таких как MS MARCO, что ограничивало обобщение на новые домены. Методы, как SimCSE (2021), не учитывали асимметрию между запросами и документами.

## Подходы
На момент появления E5 существовали:
- BM25: классический Sparse Retrieval, не учитывающий семантику.
- DPR (2020): Bi-Encoder, обученный на малом количестве золотых пар с Hard Negatives.
- Contriever (2021): Contrastive Learning на неразмеченных данных, уступает в точности без Fine-tuning.
- SimCSE (2021): оптимизирован под схожесть предложений, а не поиск.

## Идея
Двухэтапный процесс обучения с созданием датасета CCP (1.3 млрд пар) и введением префиксов `query: ` и `passage: ` для понимания асимметрии поиска.

## Архитектура
E5 использует Transformer-энкодер (версии `small`, `base`, `large` на основе BERT или RoBERTa).
1. Input: Текст с префиксом (`query: {text}`, `passage: {text}`).
2. Encoder: Обработка трансформером.
3. Pooling: Mean Pooling по всем токенам выходного слоя.
4. Normalization: Нормализация L2 для Cosine Similarity.

<img src="img/img.png" width=500>

## Обучение
1. Contrastive Pre-training: Обучение на 1.3 млрд "слабых" пар с InfoNCE loss.
2. Supervised Fine-tuning: Дообучение на высококачественных данных (MS MARCO, NLI) с Hard Negatives.

## Инференс
1. Индексация: Документы прогоняются через энкодер с префиксом `passage: ` и сохраняются в векторную БД.
2. Обработка запроса: Запрос переводится в вектор с префиксом `query: `.
3. Поиск: K-NN в векторном пространстве.

## Результаты
E5 показала прирост на бенчмарке BEIR (Zero-shot retrieval):
- E5-large достигла 51.2 nDCG@10, на 2.4 п.п. выше Contriever и на 7-10 п.п. выше моделей на MS MARCO.
- E5-small (33M параметров) превзошла DPR-base (110M параметров), подчеркивая важность предобучения на CCP.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример использования модели E5 для задачи Dense Retrieval.
# Мы будем использовать библиотеку `transformers` для загрузки предобученной модели E5.
# Для демонстрации мы создадим простую систему поиска, используя FAISS для индексации и поиска ближайших соседей.

from transformers import AutoTokenizer, AutoModel
import torch
import faiss
import numpy as np

# Загрузка предобученной модели E5 и токенизатора
model_name = "intfloat/e5-base"  # Замените на фактическое имя модели E5, если она доступна в transformers
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Функция для получения эмбеддингов текста
def get_embedding(text, prefix):
    # Добавляем префикс к тексту
    input_text = f"{prefix}: {text}"
    # Токенизация текста
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True)
    # Получение эмбеддингов
    with torch.no_grad():
        outputs = model(**inputs)
    # Используем Mean Pooling по всем токенам
    embeddings = outputs.last_hidden_state.mean(dim=1)
    # Нормализация L2
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    return embeddings.numpy()

# Пример документов и запросов
documents = [
    "Machine learning is a method of data analysis that automates analytical model building.",
    "Artificial intelligence is intelligence demonstrated by machines, in contrast to the natural intelligence displayed by humans and animals.",
    "Deep learning is part of a broader family of machine learning methods based on artificial neural networks."
]

queries = [
    "What is machine learning?",
    "Explain artificial intelligence."
]

# Получение эмбеддингов для документов
document_embeddings = np.vstack([get_embedding(doc, "passage") for doc in documents])

# Индексация документов с помощью FAISS
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Используем Inner Product, так как векторы нормализованы
index.add(document_embeddings)

# Обработка запросов и поиск ближайших соседей
for query in queries:
    query_embedding = get_embedding(query, "query")
    # Поиск K ближайших соседей
    K = 2
    distances, indices = index.search(query_embedding, K)
    print(f"Query: {query}")
    for i in range(K):
        print(f"Document {i+1}: {documents[indices[0][i]]} (Score: {distances[0][i]:.4f})")
    print()

# Этот пример демонстрирует использование модели E5 для задачи Dense Retrieval.
# Мы используем префиксы 'query:' и 'passage:' для различения запросов и документов,
# что помогает модели учитывать асимметрию между ними.
# FAISS используется для быстрого поиска ближайших соседей в векторном пространстве.